# Validation B — cooling layers from digitized spectra

This notebook runs three Figure 4d cooling calculations from digitized PDMS, SDS/PDMS, and ADS/PDMS spectra, then reproduces the digitized Figure 5d family. It does not require S4.

**Learning goals:** run several YAML experiments; compare zero-cooling-power temperatures; identify where digitized inputs and environmental substitutions limit the claim.

## 1. Prepare the temporary Colab runtime

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import shlex
import subprocess
import sys

from IPython.display import Image, Markdown, display

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("Open this notebook in Google Colab before running setup.")

PROJECT_DIR = Path("/content/radcoolpv-py")

def run_command(args: list[str], cwd: Path | None = None, capture: bool = False):
    print("$", shlex.join(args))
    return subprocess.run(
        args, cwd=cwd, check=True, text=True,
        capture_output=capture,
    )

if not PROJECT_DIR.exists():
    run_command([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/gsilvaoelker/radcoolpv-py.git",
        str(PROJECT_DIR),
    ])

run_command([
    sys.executable, "-m", "pip", "install", "--quiet", "--editable", ".",
], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)

print("Repository:", PROJECT_DIR)

## 2. Run the four YAML cases

In [ ]:
case_names = ["fig4d_pdms", "fig4d_sds", "fig4d_ads", "fig5d_cooling_family"]
for case_name in case_names:
    run_command([
        "radcoolpv", "run", f"validations/validation B/{case_name}.yaml",
    ], cwd=PROJECT_DIR)

## 3. Inspect the calculated figures and equilibrium points

In [ ]:
validation_dir = PROJECT_DIR / "validations/validation B"
rows = []
for film in ["pdms", "sds", "ads"]:
    manifests = list((validation_dir / "results" / film).glob("*/run.json"))
    path = max(manifests, key=lambda item: item.stat().st_mtime)
    scalar = json.loads(path.read_text())["thermal_results"]
    rows.append((film, scalar["equilibrium_temperature_K"]))

if rows:
    table = "| Film | Calculated zero-crossing |\n|---|---:|\n" + "\n".join(
        f"| {name.upper()} | {temperature:.2f} K |" for name, temperature in rows
    )
    display(Markdown(table))

for figure in sorted(validation_dir.glob("results/*/*/figures/*.png")):
    display(Image(filename=str(figure)))

## 4. Evidence limits and exercise

The expected Figure 4d zero crossings are approximately 329.92 K (PDMS), 328.28 K (SDS/PDMS), and 325.44 K (ADS/PDMS). The ambient temperature is an unconfirmed 298.0 K, and the bundled Cerro Pachón atmosphere substitutes for Hiroshima. Figure 5d is a reproduction of digitized curves because the commercial module's raw optical stack is unavailable; it is not an independent model validation.

**Exercise:** copy one Figure 4d YAML, change `thermal.solar_irradiance` from 800.0 to 600.0 W/m², and explain the shift in the zero crossing.

**Reference:** T. H. Le et al., *ACS Photonics* 13, 1108–1121 (2026), [doi:10.1021/acsphotonics.5c02627](https://doi.org/10.1021/acsphotonics.5c02627).